# Graphagate Training & Evaluation

Questo notebook permette di addestrare e testare il modello **Temporal Graph Network** (TGN) in locale, sfruttando l'accelerazione **Metal Performance Shaders (MPS)** dei Mac.
In questo modo non c'è bisogno di usare Docker e si possono ottenere prestazioni di training nettamente superiori sul chip M4 Pro.

In [1]:
%load_ext autoreload
%autoreload 2

import torch
import numpy as np
import sys

if torch.backends.mps.is_available():
    print("MPS accelerato (Metal) trovato. Verrà utilizzata la GPU del Mac.")
elif torch.cuda.is_available():
    print("CUDA GPU trovata.")
else:
    print("GPU non trovata, si userà la CPU.")

CUDA GPU trovata.


## 1. Configurazione Iperparametri
Qui modifichiamo i parametri per riflettere il nuovo scaling deciso nel plan (es. 1000 utenti + 1000 guest, 2000 device).

In [2]:
from graphagate.config import TGNConfig
from graphagate.train_tgn import train_tgn

cfg = TGNConfig(
    # Scale defaults up as discussed
    num_users=1000,
    num_devices=2000,
    num_sources=1500,
    num_configs=400,
    num_events=50000,   # Increase events since we have more entities
    
    # Training paramsx
    epochs=20,           # Keep low for initial testing
    batch_size=256,
    eval_batch_size=128,
    
    # Ensure memory capacity accounts for the extra 1000 guests
    capacity_headroom=2000
)

/home/gabs/Documenti/Università/Advanced Cybersecurity/Papergate/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Addestramento e Valutazione
Eseguiamo il processo completo. La pipeline includerà la generazione dei dati sintetici aggiornati (con ritmi circadiani e lateral movement chains).

In [3]:
# Patch temporanea per il bug di PyTorch MPS con int64 su scatter_reduce_
import torch
import torch_geometric.nn.models.tgn as tgn

_original_scatter = tgn.scatter
def _patched_scatter(src, index, dim=0, dim_size=None, reduce='sum'):
    if reduce in ['max', 'amax'] and src.dtype == torch.int64 and getattr(src.device, 'type', '') == 'mps':
        return _original_scatter(src.float(), index, dim, dim_size, reduce).long()
    return _original_scatter(src, index, dim, dim_size, reduce)
tgn.scatter = _patched_scatter

# Esegui il training. Verranno salvati gli artefatti in public/
metrics = train_tgn(cfg)

print("\n--- Training Completato ---")
for k, v in metrics.items():
    if isinstance(v, float):
        print(f"{k}: {v:.4f}")
    else:
        print(f"{k}: {v}")

Generating streaming data...
Using device: cuda
--- INIZIO ADDESTRAMENTO ONE-CLASS (solo traffico benigno) ---


Epoch 01/20 [train]: 100%|##########| 136/136 [00:34<00:00,  3.94it/s, loss=4.3471]


Epoch 01 | Train Loss: 5.0496


Epoch 02/20 [train]: 100%|##########| 136/136 [00:32<00:00,  4.23it/s, loss=4.1654]


Epoch 02 | Train Loss: 4.4270


...

Epoch 08 | Train Loss: 2.4022


Epoch 09/20 [train]: 100%|##########| 136/136 [00:31<00:00,  4.25it/s, loss=2.3940]


Epoch 09 | Train Loss: 2.3569


Epoch 10/20 [train]:  99%|#########8| 134/136 [00:31<00:00,  4.28it/s, loss=2.5510]


OutOfMemoryError: CUDA out of memory. Tried to allocate 862.00 MiB. GPU 0 has a total capacity of 15.47 GiB of which 563.38 MiB is free. Process 15544 has 207.79 MiB memory in use. Process 37932 has 60.16 MiB memory in use. Process 43889 has 40.28 MiB memory in use. Process 108541 has 145.48 MiB memory in use. Including non-PyTorch memory, this process has 14.00 GiB memory in use. Of the allocated memory 10.56 GiB is allocated by PyTorch, and 3.15 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)